# 01 — Dataset Understanding

## Multiclass NLP Dataset for Phishing and Social Engineering Threat Detection

Este notebook documenta a origem, a reconstrução estrutural, a análise exploratória e o tratamento inicial do dataset de segurança utilizado no Fintech Guard.

## 1. Origem e finalidade

- **Fonte:** Zenodo — https://zenodo.org/records/15235123
- **DOI:** https://doi.org/10.5281/zenodo.15235123
- **Autor institucional:** Engineering Ingegneria Informatica Spa
- **Publicação:** 17 de abril de 2025, versão v1
- **Idioma:** inglês
- **Finalidade:** classificação multiclasse de mensagens benignas e ameaças de phishing ou engenharia social.
- **Privacidade:** o registro informa que as mensagens são anonimizadas e não contêm dados pessoais.
- **Licença:** não especificada no campo de direitos do registro consultado; confirmação com os autores permanece pendente.

## 2. Importações e caminhos

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent

caminho_raw = ROOT / 'data' / 'raw' / 'security' / 'phishing_nlp_dataset.xlsx'
caminho_interim = ROOT / 'data' / 'interim' / 'security' / 'phishing_nlp_dataset.csv'
caminho_processed = ROOT / 'data' / 'processed' / 'security' / 'phishing_nlp_dataset.csv'

## 3. Reconstrução estrutural do arquivo de origem

O arquivo `.xlsx` possui 621 rótulos anexados ao final de `Corpus` por tabulação. Em outras 3 linhas, mensagens longas foram divididas entre `Corpus` e `Labels`. A célula abaixo recompõe cada linha, separa o último campo como rótulo e gera um CSV intermediário com as colunas padronizadas `text` e `category`. O arquivo original não é alterado.

In [ ]:
df_raw = pd.read_excel(caminho_raw)

linhas_recompostas = df_raw.apply(
    lambda linha: (
        str(linha['Corpus'])
        if pd.isna(linha['Labels'])
        else str(linha['Corpus']).rstrip() + ' ' + str(linha['Labels']).lstrip()
    ),
    axis=1
)

partes = linhas_recompostas.str.rsplit('\t', n=1, expand=True)
df_security = pd.DataFrame({
    'text': partes[0].str.strip(),
    'category': partes[1].str.strip()
})

caminho_interim.parent.mkdir(parents=True, exist_ok=True)
df_security.to_csv(caminho_interim, index=False)

print('Linhas reconstruídas:', len(df_security))
print('Colunas:', df_security.columns.tolist())

## 4. Estrutura dos dados

### 4.1 Dimensões, colunas e tipos

In [ ]:
print('Formato:', df_security.shape)
print('Colunas:', df_security.columns.tolist())
print('Categorias distintas:', df_security['category'].nunique())
df_security.info()

**Interpretação:** O conjunto reconstruído possui 624 registros e 2 colunas, `text` e `category`, ambas textuais e sem valores ausentes. A coluna `text` contém a mensagem que será fornecida ao classificador, enquanto `category` contém uma das 6 classes que o modelo deverá prever.

### 4.2 Amostra dos registros

In [ ]:
df_security.sample(10, random_state=42)

**Interpretação:** A amostra permite verificar qualitativamente que as mensagens estão em inglês, possuem tamanhos variados e representam tanto conteúdo benigno quanto diferentes técnicas de ameaça. A amostragem é apenas uma inspeção inicial e não substitui as verificações quantitativas das próximas seções.

## 5. Qualidade dos dados

### 5.1 Valores ausentes

In [ ]:
df_security.isna().sum()

**Interpretação:** Não foram encontrados valores ausentes em `text` ou `category`. Portanto, não há necessidade de imputação ou remoção de registros por nulidade.

### 5.2 Textos vazios ou compostos apenas por espaços

In [ ]:
textos_vazios = df_security['text'].str.strip().eq('').sum()
categorias_vazias = df_security['category'].str.strip().eq('').sum()
espacos_texto = df_security['text'].ne(df_security['text'].str.strip()).sum()
espacos_categoria = df_security['category'].ne(
    df_security['category'].str.strip()
).sum()

print('Textos vazios:', textos_vazios)
print('Categorias vazias:', categorias_vazias)
print('Textos com espaços nas extremidades:', espacos_texto)
print('Categorias com espaços nas extremidades:', espacos_categoria)

**Interpretação:** Não foram encontrados textos ou categorias vazias, nem espaços excedentes nas extremidades. A reconstrução estrutural já produziu valores preenchidos e padronizados nesse aspecto.

### 5.3 Duplicatas e conflitos de rótulo

In [ ]:
duplicatas_exatas_excedentes = df_security.duplicated().sum()
linhas_em_duplicatas_exatas = df_security.duplicated(keep=False).sum()

df_security_duplicatas = df_security.assign(
    text_normalized=df_security['text'].str.strip().str.lower()
)
mascara_duplicatas_normalizadas = (
    df_security_duplicatas['text_normalized'].duplicated(keep=False)
)
quantidade_rotulos_por_texto = (
    df_security_duplicatas
    .groupby('text_normalized')['category']
    .nunique()
)
textos_com_conflito = set(
    quantidade_rotulos_por_texto[quantidade_rotulos_por_texto > 1].index
)
conflitos_rotulo = (
    df_security_duplicatas[
        df_security_duplicatas['text_normalized'].isin(textos_com_conflito)
    ]
    .sort_values(['text_normalized', 'category'])
    [['text', 'category']]
)

print('Duplicatas exatas adicionais:', duplicatas_exatas_excedentes)
print('Linhas envolvidas em duplicatas exatas:', linhas_em_duplicatas_exatas)
print(
    'Linhas envolvidas em duplicatas normalizadas:',
    mascara_duplicatas_normalizadas.sum()
)
print('Textos normalizados com conflito de rótulo:', len(textos_com_conflito))
display(conflitos_rotulo)

**Interpretação:** Foram encontradas 9 ocorrências excedentes de duplicatas exatas, distribuídas entre 14 linhas. Ao comparar os textos após normalização temporária, 24 linhas aparecem em grupos repetidos. O ponto mais crítico é a existência de 6 textos normalizados associados a mais de uma categoria, principalmente `Phishing` e `Pretexting`. Esses conflitos podem fornecer respostas contraditórias ao classificador e precisam ser analisados antes da limpeza; nenhuma remoção é aplicada nesta etapa.

## 6. Distribuição das categorias

In [ ]:
distribuicao_categorias = df_security['category'].value_counts()

print('Quantidade de categorias:', df_security['category'].nunique())
display(distribuicao_categorias.to_frame('quantidade'))

**Interpretação:** O dataset possui 6 classes. `NOT-Malicious General Class` é a mais frequente, com 171 exemplos, e `Malware` e `Pretexting` são as menos frequentes, com 78 cada. A maior classe possui pouco mais que o dobro de exemplos das menores, indicando desbalanceamento moderado que deverá ser considerado na divisão dos dados e na avaliação do modelo.

## 7. Comprimento das mensagens

In [ ]:
comprimento_texto = df_security['text'].str.len()

display(comprimento_texto.describe())

resumo_comprimento_categoria = (
    df_security.assign(text_length=comprimento_texto)
    .groupby('category')['text_length']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .sort_values('median')
    .round(2)
)
display(resumo_comprimento_categoria)

mensagens_mais_longas = (
    df_security.loc[comprimento_texto.nlargest(5).index]
    .assign(text_length=comprimento_texto.loc[comprimento_texto.nlargest(5).index])
    [['category', 'text_length', 'text']]
)
display(mensagens_mais_longas)

**Interpretação:** O comprimento mediano é de 99 caracteres, mas a média é aproximadamente 184,62 e o máximo chega a 3.693, indicando forte assimetria à direita. As mensagens mais longas pertencem a `Phishing` e representam narrativas extensas de fraude, não erros evidentes de reconstrução. O comprimento típico também varia entre as classes; por isso, mensagens longas não devem ser removidas automaticamente.

## 8. Visualizações

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

distribuicao_categorias.sort_values().plot(
    kind='barh',
    ax=axes[0],
    color='darkorange'
)
axes[0].set_title('Distribuição das categorias')
axes[0].set_xlabel('Quantidade de mensagens')
axes[0].set_ylabel('Categoria')
axes[0].set_xlim(0, 180)
axes[0].set_xticks(range(0, 181, 20))

axes[1].hist(comprimento_texto, bins=40, color='steelblue', edgecolor='black')
axes[1].axvline(comprimento_texto.median(), color='red', linestyle='--', label='Mediana')
axes[1].set_title('Comprimento — escala completa')
axes[1].set_xlabel('Quantidade de caracteres')
axes[1].set_ylabel('Frequência')
axes[1].legend()

comprimentos_ate_600 = comprimento_texto[comprimento_texto <= 600]
axes[2].hist(
    comprimentos_ate_600,
    bins=range(0, 626, 25),
    color='seagreen',
    edgecolor='black'
)
axes[2].axvline(comprimento_texto.median(), color='red', linestyle='--', label='Mediana')
axes[2].set_title('Comprimento — detalhe até 600')
axes[2].set_xlabel('Quantidade de caracteres')
axes[2].set_ylabel('Frequência')
axes[2].set_xticks(range(0, 601, 100))
axes[2].legend()

plt.tight_layout()
plt.show()

**Interpretação:** O gráfico de categorias evidencia a maior presença de mensagens benignas e o menor volume de `Malware` e `Pretexting`. O histograma completo mostra a cauda formada por poucas mensagens muito longas; o recorte até 600 caracteres permite observar a concentração principal sem esconder que valores superiores existem. A linha tracejada indica a mediana de 99 caracteres.

## 9. Decisões e aplicação da limpeza

### 9.1 Plano de limpeza

#### Estrutura do arquivo de origem

**Problema identificado:** Os rótulos estavam anexados ao final das mensagens por tabulação e 3 textos longos estavam divididos entre as colunas do Excel.  
**Decisão:** Reunir os fragmentos, separar o último campo como rótulo e padronizar as colunas como `text` e `category`.  
**Justificativa:** A estrutura original não permitia utilizar diretamente `Corpus` e `Labels` como entrada e saída do classificador.  
**Impacto esperado:** Reconstrução dos 624 registros sem alteração do arquivo bruto.

#### Textos com conflitos de rótulo

**Problema identificado:** Foram encontrados 6 textos normalizados associados a mais de uma categoria, envolvendo 15 linhas.  
**Decisão:** Remover integralmente os grupos conflitantes do conjunto tratado.  
**Justificativa:** Escolher um rótulo sem evidência externa introduziria uma resposta arbitrária, enquanto manter categorias contraditórias prejudicaria o aprendizado supervisionado.  
**Impacto esperado:** Remoção de 15 linhas ambíguas sem inventar uma categoria correta.

#### Duplicatas sem conflito

**Problema identificado:** Após a retirada dos conflitos, ainda existem repetições do mesmo texto normalizado com a mesma categoria.  
**Decisão:** Manter a primeira ocorrência e remover as repetições excedentes.  
**Justificativa:** Exemplos idênticos não acrescentam informação e podem aumentar artificialmente o peso de determinadas mensagens.  
**Impacto esperado:** Remoção de 6 repetições excedentes.

#### Mensagens longas e nomes das categorias

**Problema identificado:** Existem mensagens de phishing com até 3.693 caracteres, e a distribuição das classes é moderadamente desbalanceada.  
**Decisão:** Preservar as mensagens longas e os nomes originais das 6 categorias. Não balancear os dados nesta etapa.  
**Justificativa:** Os textos longos representam narrativas legítimas de fraude. Balanceamento e codificação de rótulos dependem da estratégia de treinamento e devem ocorrer somente após a divisão dos dados.  
**Impacto esperado:** Preservação da diversidade textual e adiamento de decisões específicas do modelo.

### 9.2 Aplicação e validação

In [ ]:
df_security_clean = df_security[['text', 'category']].copy()
df_security_clean['text_normalized'] = (
    df_security_clean['text'].str.strip().str.lower()
)

quantidade_categorias_por_texto = (
    df_security_clean
    .groupby('text_normalized')['category']
    .nunique()
)
textos_conflitantes = set(
    quantidade_categorias_por_texto[quantidade_categorias_por_texto > 1].index
)
mascara_conflitos = df_security_clean['text_normalized'].isin(textos_conflitantes)
linhas_conflitantes_removidas = mascara_conflitos.sum()
df_security_clean = df_security_clean.loc[~mascara_conflitos].copy()

linhas_antes_deduplicacao = len(df_security_clean)
df_security_clean = df_security_clean.drop_duplicates(
    subset=['text_normalized', 'category'],
    keep='first'
)
duplicatas_removidas = linhas_antes_deduplicacao - len(df_security_clean)

df_security_clean = (
    df_security_clean
    .drop(columns='text_normalized')
    .reset_index(drop=True)
)

text_normalized_validacao = df_security_clean['text'].str.strip().str.lower()
conflitos_restantes = (
    df_security_clean.assign(text_normalized=text_normalized_validacao)
    .groupby('text_normalized')['category']
    .nunique()
    .gt(1)
    .sum()
)
duplicatas_restantes = (
    df_security_clean.assign(text_normalized=text_normalized_validacao)
    .duplicated(subset=['text_normalized', 'category'])
    .sum()
)

print('Linhas originais reconstruídas:', len(df_security))
print('Linhas conflitantes removidas:', linhas_conflitantes_removidas)
print('Duplicatas excedentes removidas:', duplicatas_removidas)
print('Formato tratado:', df_security_clean.shape)
print('Conflitos restantes:', conflitos_restantes)
print('Duplicatas normalizadas restantes:', duplicatas_restantes)
print('Valores ausentes:', df_security_clean.isna().sum().to_dict())
print('Categorias preservadas:', df_security_clean['category'].nunique())

**Interpretação:** Foram removidas 15 linhas pertencentes a grupos com rótulos contraditórios e 6 repetições excedentes sem conflito. O conjunto tratado possui 603 mensagens, mantém as 6 categorias e não apresenta valores ausentes, duplicatas normalizadas ou conflitos de rótulo. Os textos originais dos registros mantidos foram preservados; `lower()` foi utilizado somente como chave temporária de comparação.

## 10. Exportação dos dados tratados

In [ ]:
caminho_processed.parent.mkdir(parents=True, exist_ok=True)
df_security_clean.to_csv(caminho_processed, index=False)

df_security_exportado = pd.read_csv(caminho_processed)

print('Arquivo exportado:', caminho_processed)
print('Formato:', df_security_exportado.shape)
print('Colunas:', df_security_exportado.columns.tolist())
print('Categorias:', df_security_exportado['category'].nunique())
print('Valores ausentes:', df_security_exportado.isna().sum().to_dict())

**Interpretação:** O dataset tratado foi exportado em CSV para `data/processed/security`, sem o índice do pandas. A nova leitura confirma 603 registros, 2 colunas, 6 categorias e ausência de valores nulos.

## 11. Conclusões

**Principais características encontradas:** O dataset reconstruído possui 624 mensagens em inglês e 6 classes: `Phishing`, `Malware`, `Scareware`, `Baiting`, `Pretexting` e `NOT-Malicious General Class`. A classe benigna é a mais frequente, com 171 exemplos, enquanto `Malware` e `Pretexting` possuem 78 cada. O comprimento mediano é de 99 caracteres, mas mensagens de phishing chegam a 3.693 caracteres.  

**Problemas de qualidade identificados:** O arquivo de origem apresentava rótulos anexados aos textos e 3 mensagens divididas entre colunas. Após a reconstrução, foram encontradas 9 duplicatas exatas adicionais e 6 textos normalizados associados a categorias diferentes, envolvendo 15 linhas. Não foram encontrados valores ausentes, textos vazios ou espaços excedentes nas extremidades.  

**Transformações realizadas:** A estrutura do Excel foi reconstruída nas colunas `text` e `category`. Os grupos com rótulos conflitantes foram removidos integralmente e as repetições excedentes restantes foram deduplicadas. Mensagens longas, pontuação, capitalização e nomes das categorias foram preservados. O resultado final possui 603 registros e foi exportado para `data/processed/security/phishing_nlp_dataset.csv`.  

**Limitações da análise:** O registro do Zenodo não apresenta uma licença explícita no campo de direitos. O dataset é pequeno, moderadamente desbalanceado e contém sobreposição semântica entre `Phishing` e `Pretexting`. A inspeção manual não foi exaustiva para todas as mensagens.  

**Próximos passos:** Confirmar a licença com os responsáveis pelo dataset, realizar uma divisão estratificada em treino e teste, definir o pré-processamento textual, avaliar técnicas de balanceamento apenas no treino e comparar modelos por Accuracy, Precision, Recall, F1-score e matriz de confusão.